# 01 Data Collection

In this notebook, we collect historical Formula 1 race data from 2018 onward.

The goal of this step is to create a clean raw dataset that will later be used for:

- feature engineering
- machine learning model training
- race result prediction
- driver performance analysis
- driver profile cards

We will use the Jolpica F1 API, which provides Formula 1 data in an Ergast-compatible format.

# Import Libs

In [14]:
import requests
import pandas as pd
import time 
from pathlib import Path

# Create Project Paths

Save collected data in data/raw

In [15]:
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Set API Parametrs 

define URL of API and seasons we want collect

In [16]:
BASE_URL = "https://api.jolpi.ca/ergast/f1"

START_YEAR = 2001
END_YEAR = 2025

SEASONS = list(range(START_YEAR, END_YEAR +1))

SEASONS

[2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025]

# Create API request function

get url, return json

In [17]:
import requests
import time

def get_json(url, max_retries=5, timeout=60):
    """
    Send GET request to the API with retries.
    
    If the API is slow or temporarily unavailable, the function waits
    and tries again instead of stopping the whole notebook.
    """
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Request attempt {attempt}: {url}")
            
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            
            return response.json()
        
        except requests.exceptions.Timeout:
            print(f"Timeout on attempt {attempt}. Waiting before retry...")
            time.sleep(5)
        
        except requests.exceptions.RequestException as error:
            print(f"Request failed on attempt {attempt}")
            print(f"Error: {error}")
            time.sleep(5)
    
    print(f"Failed after {max_retries} attempts: {url}")
    return None

# Test API Connection

In [30]:
test_url = f"{BASE_URL}/2001/1/results.json"
test_data = get_json(test_url)

test_data.keys() if test_data else None

Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/1/results.json


dict_keys(['MRData'])

In [31]:
test_data["MRData"].keys()

dict_keys(['xmlns', 'series', 'url', 'limit', 'offset', 'total', 'RaceTable'])

In [32]:
test_data["MRData"]["RaceTable"].keys()

dict_keys(['season', 'round', 'Races'])

In [33]:
len(test_data["MRData"]["RaceTable"]["Races"])

1

In [22]:
def get_season_rounds(season):
    """
    Get number of races in a season.
    """
    url = f"{BASE_URL}/{season}.json"
    data = get_json(url)
    
    if data is None:
        return []
    
    races = data["MRData"]["RaceTable"]["Races"]
    rounds = [int(race["round"]) for race in races]
    
    return rounds

In [34]:
rounds_2001 = get_season_rounds(2001)
rounds_2001

Request attempt 1: https://api.jolpi.ca/ergast/f1/2001.json


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

## 6. Collect Race Results

Now we collect race results for each season.

For every race, we extract:

- season
- round
- race name
- race date
- circuit information
- driver information
- constructor information
- grid position
- final position
- points
- laps
- race status

This table will become the base dataset for our machine learning project.

In [35]:
def collect_race_results(seasons):
    """
    Collect Formula 1 race results race by race.
    
    This approach is more stable than requesting the whole season at once.
    """
    
    all_results = []
    
    for season in seasons:
        print(f"\nCollecting season {season}...")
        
        rounds = get_season_rounds(season)
        
        if not rounds:
            print(f"No rounds found for season {season}")
            continue
        
        for round_number in rounds:
            print(f"Collecting season {season}, round {round_number}...")
            
            url = f"{BASE_URL}/{season}/{round_number}/results.json"
            data = get_json(url)
            
            if data is None:
                continue
            
            races = data["MRData"]["RaceTable"]["Races"]
            
            if len(races) == 0:
                continue
            
            race = races[0]
            
            season_year = int(race["season"])
            round_number = int(race["round"])
            race_name = race["raceName"]
            race_date = race.get("date")
            
            circuit = race["Circuit"]
            circuit_id = circuit["circuitId"]
            circuit_name = circuit["circuitName"]
            circuit_country = circuit["Location"]["country"]
            circuit_locality = circuit["Location"]["locality"]
            
            for result in race["Results"]:
                driver = result["Driver"]
                constructor = result["Constructor"]
                
                row = {
                    "season": season_year,
                    "round": round_number,
                    "race_name": race_name,
                    "race_date": race_date,
                    
                    "circuit_id": circuit_id,
                    "circuit_name": circuit_name,
                    "circuit_country": circuit_country,
                    "circuit_locality": circuit_locality,
                    
                    "driver_id": driver["driverId"],
                    "driver_code": driver.get("code"),
                    "driver_number": driver.get("permanentNumber"),
                    "driver_given_name": driver["givenName"],
                    "driver_family_name": driver["familyName"],
                    "driver_nationality": driver["nationality"],
                    
                    "constructor_id": constructor["constructorId"],
                    "constructor_name": constructor["name"],
                    "constructor_nationality": constructor["nationality"],
                    
                    "grid": result.get("grid"),
                    "position": result.get("position"),
                    "position_text": result.get("positionText"),
                    "points": result.get("points"),
                    "laps": result.get("laps"),
                    "status": result.get("status")
                }
                
                all_results.append(row)
            
            time.sleep(1)
    
    return pd.DataFrame(all_results)

In [36]:
SEASONS_TEST = [2001]

race_results_df = collect_race_results(SEASONS_TEST)

race_results_df.head()


Request attempt 1: https://api.jolpi.ca/ergast/f1/2001.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/1/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/2/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/3/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/4/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/5/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/6/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/7/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/8/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/9/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/10/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/11/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/12/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/13/results.json
Request attempt 1: https:

,season,round,race_name,race_date,circuit_id,circuit_name,circuit_country,circuit_locality,driver_id,driver_code,...,driver_nationality,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status
0,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,michael_schumacher,MSC,...,German,ferrari,Ferrari,Italian,1,1,1,10,58,Finished
1,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,coulthard,COU,...,British,mclaren,McLaren,British,6,2,2,6,58,Finished
2,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,barrichello,BAR,...,Brazilian,ferrari,Ferrari,Italian,2,3,3,4,58,Finished
3,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,heidfeld,HEI,...,German,sauber,Sauber,Swiss,10,4,4,3,58,Finished
4,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,frentzen,None,...,German,jordan,Jordan,Irish,4,5,5,2,58,Finished


In [37]:
race_results_df.shape

(373, 23)

# Save collected data

In [38]:
race_results_df.to_csv("../data/raw/race_results_2001_test.csv", index=False)

# Collect data from other years

In [39]:
SEASONS = list(range(2001, 2026))

race_results_df = collect_race_results(SEASONS)

race_results_df.head()


Request attempt 1: https://api.jolpi.ca/ergast/f1/2001.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/1/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/2/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/3/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/4/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/5/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/6/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/7/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/8/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/9/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/10/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/11/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/12/results.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/13/results.json
Request attempt 1: https:

,season,round,race_name,race_date,circuit_id,circuit_name,circuit_country,circuit_locality,driver_id,driver_code,...,driver_nationality,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status
0,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,michael_schumacher,MSC,...,German,ferrari,Ferrari,Italian,1,1,1,10,58,Finished
1,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,coulthard,COU,...,British,mclaren,McLaren,British,6,2,2,6,58,Finished
2,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,barrichello,BAR,...,Brazilian,ferrari,Ferrari,Italian,2,3,3,4,58,Finished
3,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,heidfeld,HEI,...,German,sauber,Sauber,Swiss,10,4,4,3,58,Finished
4,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,frentzen,None,...,German,jordan,Jordan,Irish,4,5,5,2,58,Finished


# Save results from 2001 - 2026

In [40]:
race_results_path = "../data/raw/race_results_2001_2026.csv"

race_results_df.to_csv(race_results_path, index=False)

race_results_check = pd.read_csv(race_results_path)

race_results_check.head()

,season,round,race_name,race_date,circuit_id,circuit_name,circuit_country,circuit_locality,driver_id,driver_code,...,driver_nationality,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status
0,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,michael_schumacher,MSC,...,German,ferrari,Ferrari,Italian,1,1,1,10.0,58,Finished
1,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,coulthard,COU,...,British,mclaren,McLaren,British,6,2,2,6.0,58,Finished
2,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,barrichello,BAR,...,Brazilian,ferrari,Ferrari,Italian,2,3,3,4.0,58,Finished
3,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,heidfeld,HEI,...,German,sauber,Sauber,Swiss,10,4,4,3.0,58,Finished
4,2001,1,Australian Grand Prix,2001-03-04,albert_park,Albert Park Grand Prix Circuit,Australia,Melbourne,frentzen,NaN,...,German,jordan,Jordan,Irish,4,5,5,2.0,58,Finished


# Check values

In [41]:
race_results_check.shape

(10177, 23)

In [42]:
race_results_check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10177 entries, 0 to 10176
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   season                   10177 non-null  int64  
 1   round                    10177 non-null  int64  
 2   race_name                10177 non-null  object 
 3   race_date                10177 non-null  object 
 4   circuit_id               10177 non-null  object 
 5   circuit_name             10177 non-null  object 
 6   circuit_country          10177 non-null  object 
 7   circuit_locality         10177 non-null  object 
 8   driver_id                10177 non-null  object 
 9   driver_code              9737 non-null   object 
 10  driver_number            7023 non-null   float64
 11  driver_given_name        10177 non-null  object 
 12  driver_family_name       10177 non-null  object 
 13  driver_nationality       10177 non-null  object 
 14  constructor_id        

In [43]:
race_results_check.groupby("season")["race_name"].nunique()

season
2001    17
2002    17
2003    16
2004    18
2005    19
2006    18
2007    17
2008    18
2009    17
2010    19
2011    19
2012    20
2013    19
2014    19
2015    19
2016    21
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
Name: race_name, dtype: int64

In [44]:
race_results_check.groupby(["season", "race_name"]).size().head(30)

season  race_name               
2001    Australian Grand Prix       22
        Austrian Grand Prix         22
        Belgian Grand Prix          22
        Brazilian Grand Prix        22
        British Grand Prix          21
        Canadian Grand Prix         22
        European Grand Prix         22
        French Grand Prix           22
        German Grand Prix           22
        Hungarian Grand Prix        22
        Italian Grand Prix          22
        Japanese Grand Prix         22
        Malaysian Grand Prix        22
        Monaco Grand Prix           22
        San Marino Grand Prix       22
        Spanish Grand Prix          22
        United States Grand Prix    22
2002    Australian Grand Prix       22
        Austrian Grand Prix         22
        Belgian Grand Prix          20
        Brazilian Grand Prix        22
        British Grand Prix          21
        Canadian Grand Prix         22
        European Grand Prix         22
        French Grand Prix      

# Check missing races

In [45]:
races_per_season = (
    race_results_check
    .groupby("season")["race_name"]
    .nunique()
    .reset_index(name="number_of_races")
)

races_per_season

,season,number_of_races
0,2001,17
1,2002,17
2,2003,16
3,2004,18
4,2005,19
5,2006,18
6,2007,17
7,2008,18
8,2009,17
9,2010,19


In [46]:
races_per_season.sort_values("season")

,season,number_of_races
0,2001,17
1,2002,17
2,2003,16
3,2004,18
4,2005,19
5,2006,18
6,2007,17
7,2008,18
8,2009,17
9,2010,19


## Check Number of Drivers per Race

In [47]:
drivers_per_race = (
    race_results_check
    .groupby(["season", "round", "race_name"])["driver_id"]
    .nunique()
    .reset_index(name="number_of_drivers")
)

drivers_per_race.head()

,season,round,race_name,number_of_drivers
0,2001,1,Australian Grand Prix,22
1,2001,2,Malaysian Grand Prix,22
2,2001,3,Brazilian Grand Prix,22
3,2001,4,San Marino Grand Prix,22
4,2001,5,Spanish Grand Prix,22


In [48]:
drivers_per_race[drivers_per_race["number_of_drivers"] < 15]

,season,round,race_name,number_of_drivers


# Check Duplicates

In [49]:
duplicates = race_results_check.duplicated(
    subset=["season", "round", "driver_id"],
    keep=False
)

race_results_check[duplicates]

,season,round,race_name,race_date,circuit_id,circuit_name,circuit_country,circuit_locality,driver_id,driver_code,...,driver_nationality,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status


In [50]:
race_results_check.isnull().sum()

season                        0
round                         0
race_name                     0
race_date                     0
circuit_id                    0
circuit_name                  0
circuit_country               0
circuit_locality              0
driver_id                     0
driver_code                 440
driver_number              3154
driver_given_name             0
driver_family_name            0
driver_nationality            0
constructor_id                0
constructor_name              0
constructor_nationality       0
grid                          0
position                      0
position_text                 0
points                        0
laps                          0
status                        0
dtype: int64

In [51]:
important_columns = [
    "season",
    "round",
    "race_name",
    "circuit_id",
    "driver_id",
    "constructor_id",
    "grid",
    "position",
    "points",
    "status"
]

race_results_check[important_columns].isnull().sum()

season            0
round             0
race_name         0
circuit_id        0
driver_id         0
constructor_id    0
grid              0
position          0
points            0
status            0
dtype: int64

In [52]:
numeric_columns = ["season", "round", "grid", "position", "points", "laps"]

for col in numeric_columns:
    race_results_check[col] = pd.to_numeric(race_results_check[col], errors="coerce")

race_results_check.dtypes

season                       int64
round                        int64
race_name                   object
race_date                   object
circuit_id                  object
circuit_name                object
circuit_country             object
circuit_locality            object
driver_id                   object
driver_code                 object
driver_number              float64
driver_given_name           object
driver_family_name          object
driver_nationality          object
constructor_id              object
constructor_name            object
constructor_nationality     object
grid                         int64
position                     int64
position_text               object
points                     float64
laps                         int64
status                      object
dtype: object

# Save validated raw version

In [53]:
validated_race_results_path = RAW_DATA_DIR / "race_results_2001_2026_validated.csv"

race_results_check.to_csv(validated_race_results_path, index=False)

validated_race_results_path

WindowsPath('../data/raw/race_results_2001_2026_validated.csv')

# Collect Qualifying Results Race by Race

In [54]:
def collect_qualifying_results(seasons):
    """
    Collect Formula 1 qualifying results race by race.
    This method is more stable than collecting the whole season at once.
    """
    
    all_qualifying = []
    
    for season in seasons:
        print(f"\nCollecting qualifying results for season {season}...")
        
        rounds = get_season_rounds(season)
        
        if not rounds:
            print(f"No rounds found for season {season}")
            continue
        
        for round_number in rounds:
            print(f"Collecting qualifying: season {season}, round {round_number}...")
            
            url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"
            data = get_json(url)
            
            if data is None:
                continue
            
            races = data["MRData"]["RaceTable"]["Races"]
            
            if len(races) == 0:
                continue
            
            race = races[0]
            
            season_year = int(race["season"])
            round_number = int(race["round"])
            race_name = race["raceName"]
            
            circuit = race["Circuit"]
            circuit_id = circuit["circuitId"]
            circuit_name = circuit["circuitName"]
            
            for result in race["QualifyingResults"]:
                driver = result["Driver"]
                constructor = result["Constructor"]
                
                row = {
                    "season": season_year,
                    "round": round_number,
                    "race_name": race_name,
                    "circuit_id": circuit_id,
                    "circuit_name": circuit_name,
                    
                    "driver_id": driver["driverId"],
                    "driver_code": driver.get("code"),
                    "driver_name": driver["givenName"] + " " + driver["familyName"],
                    
                    "constructor_id": constructor["constructorId"],
                    "constructor_name": constructor["name"],
                    
                    "qualifying_position": result.get("position"),
                    "q1": result.get("Q1"),
                    "q2": result.get("Q2"),
                    "q3": result.get("Q3")
                }
                
                all_qualifying.append(row)
            
            time.sleep(1)
    
    return pd.DataFrame(all_qualifying)

In [55]:
qualifying_test_df = collect_qualifying_results([2001])

qualifying_test_df.head()


Request attempt 1: https://api.jolpi.ca/ergast/f1/2001.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/1/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/2/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/3/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/4/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/5/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/6/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/7/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/8/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/9/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/10/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/11/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/12/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/13/qu

,season,round,race_name,circuit_id,circuit_name,driver_id,driver_code,driver_name,constructor_id,constructor_name,qualifying_position,q1,q2,q3
0,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,michael_schumacher,MSC,Michael Schumacher,ferrari,Ferrari,1,1:11.708,None,None
1,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,ralf_schumacher,SCH,Ralf Schumacher,williams,Williams,2,1:11.986,None,None
2,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,montoya,MON,Juan Pablo Montoya,williams,Williams,3,1:12.252,None,None
3,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,hakkinen,None,Mika Häkkinen,mclaren,McLaren,4,1:12.309,None,None
4,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,barrichello,BAR,Rubens Barrichello,ferrari,Ferrari,5,1:12.327,None,None


In [56]:
qualifying_test_df.shape

(22, 14)

In [57]:
qualifying_df = collect_qualifying_results(SEASONS)

qualifying_df.head()


Request attempt 1: https://api.jolpi.ca/ergast/f1/2001.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/1/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/2/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/3/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/4/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/5/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/6/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/7/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/8/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/9/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/10/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/11/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/12/qualifying.json
Request attempt 1: https://api.jolpi.ca/ergast/f1/2001/13/qu

,season,round,race_name,circuit_id,circuit_name,driver_id,driver_code,driver_name,constructor_id,constructor_name,qualifying_position,q1,q2,q3
0,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,michael_schumacher,MSC,Michael Schumacher,ferrari,Ferrari,1,1:11.708,None,None
1,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,ralf_schumacher,SCH,Ralf Schumacher,williams,Williams,2,1:11.986,None,None
2,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,montoya,MON,Juan Pablo Montoya,williams,Williams,3,1:12.252,None,None
3,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,hakkinen,None,Mika Häkkinen,mclaren,McLaren,4,1:12.309,None,None
4,2001,16,United States Grand Prix,indianapolis,Indianapolis Motor Speedway,barrichello,BAR,Rubens Barrichello,ferrari,Ferrari,5,1:12.327,None,None


# Save qualifying data

In [58]:
qualifying_df["qualifying_position"] = pd.to_numeric(
    qualifying_df["qualifying_position"],
    errors="coerce"
)

qualifying_path = RAW_DATA_DIR / "qualifying_results_2001_2026.csv"

qualifying_df.to_csv(qualifying_path, index=False)

qualifying_path

WindowsPath('../data/raw/qualifying_results_2001_2026.csv')

# Collect Race Schedule and Circuit Data

In [59]:
def collect_race_schedule(seasons):
    """
    Collect race schedule and circuit information for multiple seasons.
    
    This function collects:
    - season
    - round
    - race name
    - race date
    - circuit id
    - circuit name
    - circuit location
    - country
    
    Parameters:
        seasons (list): list of seasons
        
    Returns:
        pandas.DataFrame: race schedule and circuit data
    """
    
    all_races = []
    
    for season in seasons:
        print(f"\nCollecting race schedule for season {season}...")
        
        url = f"{BASE_URL}/{season}.json"
        data = get_json(url)
        
        if data is None:
            print(f"No data for season {season}")
            continue
        
        races = data["MRData"]["RaceTable"]["Races"]
        
        for race in races:
            circuit = race["Circuit"]
            location = circuit["Location"]
            
            row = {
                "season": int(race["season"]),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "race_date": race.get("date"),
                "race_time": race.get("time"),
                
                "circuit_id": circuit["circuitId"],
                "circuit_name": circuit["circuitName"],
                "circuit_url": circuit.get("url"),
                
                "locality": location.get("locality"),
                "country": location.get("country"),
                "lat": location.get("lat"),
                "long": location.get("long")
            }
            
            all_races.append(row)
        
        time.sleep(1)
    
    return pd.DataFrame(all_races)

In [60]:
schedule_df = collect_race_schedule(SEASONS)

schedule_df.head()


Request attempt 1: https://api.jolpi.ca/ergast/f1/2001.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2002.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2003.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2004.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2005.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2006.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2007.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2008.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2009.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2010.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2011.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2012.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2013.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2014.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2015.json

Request attempt 1: https://api.jolpi.ca/ergast/f1/2016.json

Request attempt 1: http

,season,round,race_name,race_date,race_time,circuit_id,circuit_name,circuit_url,locality,country,lat,long
0,2001,1,Australian Grand Prix,2001-03-04,None,albert_park,Albert Park Grand Prix Circuit,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Melbourne,Australia,-37.8497,144.968
1,2001,2,Malaysian Grand Prix,2001-03-18,None,sepang,Sepang International Circuit,https://en.wikipedia.org/wiki/Sepang_Internati...,Kuala Lumpur,Malaysia,2.76083,101.738
2,2001,3,Brazilian Grand Prix,2001-04-01,None,interlagos,Autódromo José Carlos Pace,https://en.wikipedia.org/wiki/Interlagos_Circuit,São Paulo,Brazil,-23.7036,-46.6997
3,2001,4,San Marino Grand Prix,2001-04-15,None,imola,Autodromo Enzo e Dino Ferrari,https://en.wikipedia.org/wiki/Imola_Circuit,Imola,Italy,44.3439,11.7167
4,2001,5,Spanish Grand Prix,2001-04-29,None,catalunya,Circuit de Barcelona-Catalunya,https://en.wikipedia.org/wiki/Circuit_de_Barce...,Barcelona,Spain,41.57,2.26111


In [61]:
schedule_path = RAW_DATA_DIR / "race_schedule_2001_2026.csv"

schedule_df.to_csv(schedule_path, index=False)

schedule_path

WindowsPath('../data/raw/race_schedule_2001_2026.csv')